<a href="https://colab.research.google.com/github/MysaniThiya/Agentra-RajarataUniversityOfSriLanka/blob/feature%2Femail-analysis/email_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git --version

git version 2.43.0


In [2]:
!git clone https://github.com/MysaniThiya/Agentra-RajarataUniversityOfSriLanka.git


Cloning into 'Agentra-RajarataUniversityOfSriLanka'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 93 (delta 37), reused 44 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 95.17 KiB | 4.53 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [3]:
%cd Agentra-RajarataUniversityOfSriLanka


/content/Agentra-RajarataUniversityOfSriLanka


In [4]:
!ls


Agentra-RajarataUniversityOfSriLanka  project		README.md
docs				      project-demo.mp4


In [5]:
!git fetch origin

In [6]:
!git branch -a

* main
  remotes/origin/Fasna
  remotes/origin/HEAD -> origin/main
  remotes/origin/feature/email-analysis
  remotes/origin/main
  remotes/origin/url-analysis-agent


In [7]:
!git checkout -b feature/email-analysis origin/feature/email-analysis

branch 'feature/email-analysis' set up to track 'origin/feature/email-analysis'.
Switched to a new branch 'feature/email-analysis'


In [8]:
!git branch


* feature/email-analysis
  main


In [9]:
import os

os.makedirs("datasets", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)
os.makedirs("models", exist_ok=True)

In [10]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib datasets

In [11]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [12]:
from datasets import load_dataset

dataset_url = (
    "https://huggingface.co/datasets/"
    "locuoco/the-biggest-spam-ham-phish-email-dataset-300000/"
    "resolve/main/df.csv"
)

# Load the CSV as a streaming dataset
email_stream = load_dataset(
    "csv",
    data_files=dataset_url,
    split="train",
    streaming=True
)

# Shuffle and select 60,000 records for the first experiment
email_stream = email_stream.shuffle(
    seed=42,
    buffer_size=10000
)

records = list(email_stream.take(60000))

# Convert records into a DataFrame
df = pd.DataFrame(records)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (60000, 2)


,label,text
0,0,daren i know it ' s first of the month but the...
1,0,jodi chris can set up a conference call with m...
2,0,the recruiting season has officially begun the...
3,0,attached is a memo summarizing policy changes ...
4,0,please let me know if you are available for a ...


In [13]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nLabel distribution:")
print(df["label"].value_counts().sort_index())

print("\nEmail examples:")
display(df[["text", "label"]].head(10))

Column names:
['label', 'text']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   60000 non-null  int64 
 1   text    60000 non-null  object
dtypes: int64(1), object(1)
memory usage: 937.6+ KB

Label distribution:
label
0    32329
2    27671
Name: count, dtype: int64

Email examples:


,text,label
0,daren i know it ' s first of the month but the...,0
1,jodi chris can set up a conference call with m...,0
2,the recruiting season has officially begun the...,0
3,attached is a memo summarizing policy changes ...,0
4,please let me know if you are available for a ...,0
5,louise effective escapenumber escapenumber esc...,0
6,new power plant added to pnm system pnm public...,0
7,iv friends i spoken with most of you over the ...,0
8,sheila changes to the manual list for north am...,0
9,i also have an opportunity for a manager top c...,0


In [14]:
label_mapping = {
    0: "Ham / Safe",
    1: "Phishing / Malicious",
    2: "Spam / Provisional Suspicious"
}

print("Original label distribution:")

for label, count in df["label"].value_counts().sort_index().items():
    print(label_mapping.get(int(label), "Unknown"), ":", count)

Original label distribution:
Ham / Safe : 32329
Spam / Provisional Suspicious : 27671


In [15]:
from datasets import load_dataset
from collections import Counter

dataset_url = (
    "https://huggingface.co/datasets/"
    "locuoco/the-biggest-spam-ham-phish-email-dataset-300000/"
    "resolve/main/df.csv"
)

# Load the complete dataset as a stream
full_stream = load_dataset(
    "csv",
    data_files=dataset_url,
    split="train",
    streaming=True
)

# Count every original label
label_counts = Counter()

for row in full_stream:
    label_counts[row["label"]] += 1

print("Complete dataset label distribution:")

for label, count in sorted(label_counts.items()):
    print(f"Label {label}: {count:,} emails")

Complete dataset label distribution:
Label 0: 168,455 emails
Label 1: 42,845 emails
Label 2: 154,148 emails


In [16]:

from datasets import load_dataset
from collections import defaultdict
import pandas as pd

dataset_url = (
    "https://huggingface.co/datasets/"
    "locuoco/the-biggest-spam-ham-phish-email-dataset-300000/"
    "resolve/main/df.csv"
)

# Load the dataset as a stream
stream = load_dataset(
    "csv",
    data_files=dataset_url,
    split="train",
    streaming=True
)

# Store up to 15,000 emails from each class
samples = defaultdict(list)

MAX_PER_CLASS = 15000

for row in stream:
    label = int(row["label"])

    if label in [0, 1, 2]:
        if len(samples[label]) < MAX_PER_CLASS:
            samples[label].append({
                "text": row["text"],
                "label": label
            })

    # Stop when all three classes have 15,000 emails
    if all(len(samples[i]) >= MAX_PER_CLASS for i in [0, 1, 2]):
        break

# Create a balanced DataFrame
balanced_df = pd.DataFrame(
    [item for label in [0, 1, 2] for item in samples[label]]
)

# Shuffle the dataset
balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Balanced dataset shape:", balanced_df.shape)

print("\nClass distribution:")
print(balanced_df["label"].value_counts().sort_index())

print("\nSample emails:")
display(balanced_df.head())


Balanced dataset shape: (45000, 2)

Class distribution:
label
0    15000
1    15000
2    15000
Name: count, dtype: int64

Sample emails:


,text,label
0,downloadable software http cabins jetlow com a...,2
1,tanya fyi this is the meeting you will be atte...,0
2,dear user welcome to the world of filehunter a...,2
3,louise this draft is very preliminary we have ...,0
4,b hf v bescapenumber bescapenumber nhkl w n h...,2


In [17]:

# STEP 1: Dataset quality checking

import pandas as pd

# Create a copy of the balanced dataset
df = balanced_df.copy()

# 1. Check missing values
print("Missing values:")
print(df.isnull().sum())

# 2. Check empty email texts
empty_texts = df["text"].astype(str).str.strip().eq("").sum()
print("\nEmpty email texts:", empty_texts)

# 3. Check exact duplicate records
duplicate_rows = df.duplicated().sum()
print("Duplicate records:", duplicate_rows)

# 4. Check duplicate email texts
duplicate_texts = df["text"].duplicated().sum()
print("Duplicate email texts:", duplicate_texts)

# 5. Check whether identical texts have conflicting labels
label_counts_per_text = df.groupby("text")["label"].nunique()
conflicting_texts = (label_counts_per_text > 1).sum()

print("Emails with conflicting labels:", conflicting_texts)

# 6. Check email text lengths
df["text_length"] = df["text"].astype(str).apply(len)

print("\nEmail text length statistics:")
print(df["text_length"].describe())

# 7. Display class distribution
print("\nClass distribution:")
print(df["label"].value_counts().sort_index())


Missing values:
text     0
label    0
dtype: int64

Empty email texts: 1
Duplicate records: 0
Duplicate email texts: 1
Emails with conflicting labels: 1

Email text length statistics:
count     45000.000000
mean       1415.320378
std        6861.123090
min           1.000000
25%         308.000000
50%         630.000000
75%        1468.000000
max      916044.000000
Name: text_length, dtype: float64

Class distribution:
label
0    15000
1    15000
2    15000
Name: count, dtype: int64


In [18]:

# STEP 2: Investigate problematic email records

# Find empty emails
empty_rows = df[
    df["text"].astype(str).str.strip().eq("")
]

print("EMPTY EMAIL RECORDS:")
display(empty_rows)

# Find duplicate email texts
duplicate_rows = df[
    df["text"].duplicated(keep=False)
].sort_values("text")

print("\nDUPLICATE EMAIL RECORDS:")
display(duplicate_rows[["label", "text"]])

# Find conflicting labels
conflicting_texts = (
    df.groupby("text")["label"]
    .nunique()
)

conflicting_texts = conflicting_texts[
    conflicting_texts > 1
].index

conflict_records = df[
    df["text"].isin(conflicting_texts)
]

print("\nCONFLICTING LABEL RECORDS:")
display(conflict_records[["label", "text"]])

# Find unusually long emails
long_emails = df[
    df["text"].str.len() > 20000
]

print("\nEMAILS LONGER THAN 20,000 CHARACTERS:")
print("Number of long emails:", len(long_emails))

display(
    long_emails[["label", "text_length"]].head(10)
)


EMPTY EMAIL RECORDS:


,text,label,text_length
43489,,1,1



DUPLICATE EMAIL RECORDS:


,label,text
3868,0,unsubscribe
7954,2,unsubscribe



CONFLICTING LABEL RECORDS:


,label,text
3868,0,unsubscribe
7954,2,unsubscribe



EMAILS LONGER THAN 20,000 CHARACTERS:
Number of long emails: 114


,label,text_length
214,0,21520
320,0,21702
1603,0,139890
1967,1,22726
2532,0,32943
2737,0,37819
2861,0,73326
3614,0,89609
3884,0,30030
3960,0,75155


In [19]:

# STEP 3: Final dataset cleaning

# Keep a copy of the original dataset
cleaned_df = df[["text", "label"]].copy()

# 1. Remove empty email texts
cleaned_df = cleaned_df[
    cleaned_df["text"].notna() &
    cleaned_df["text"].astype(str).str.strip().ne("")
]

# 2. Remove emails with conflicting labels
conflicting_texts = (
    cleaned_df.groupby("text")["label"]
    .nunique()
)

conflicting_texts = conflicting_texts[
    conflicting_texts > 1
].index

cleaned_df = cleaned_df[
    ~cleaned_df["text"].isin(conflicting_texts)
]

# 3. Remove remaining exact duplicate emails
cleaned_df = cleaned_df.drop_duplicates(
    subset=["text"],
    keep="first"
)

# Reset index
cleaned_df = cleaned_df.reset_index(drop=True)

# Display final results
print("FINAL CLEANED DATASET")
print("----------------------")

print("Dataset shape:", cleaned_df.shape)

print("\nClass distribution:")
print(
    cleaned_df["label"]
    .value_counts()
    .sort_index()
)

print("\nMissing values:")
print(cleaned_df.isnull().sum())

print("\nDuplicate emails:", cleaned_df["text"].duplicated().sum())

print("\nEmpty emails:",
      cleaned_df["text"].astype(str).str.strip().eq("").sum())


FINAL CLEANED DATASET
----------------------
Dataset shape: (44997, 2)

Class distribution:
label
0    14999
1    14999
2    14999
Name: count, dtype: int64

Missing values:
text     0
label    0
dtype: int64

Duplicate emails: 0

Empty emails: 0


In [20]:

# STEP 4: Train-Test Split

from sklearn.model_selection import train_test_split

# Separate input features and target labels
X = cleaned_df["text"]
y = cleaned_df["label"]

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display dataset sizes
print("TRAINING AND TESTING DATASET")
print("-----------------------------")

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting class distribution:")
print(y_test.value_counts().sort_index())

print("\nTotal samples:", len(X_train) + len(X_test))


TRAINING AND TESTING DATASET
-----------------------------
Training samples: 35997
Testing samples: 9000

Training class distribution:
label
0    11999
1    11999
2    11999
Name: count, dtype: int64

Testing class distribution:
label
0    3000
1    3000
2    3000
Name: count, dtype: int64

Total samples: 44997
